# Grad-CAM Attention Visualizations

In [1]:
import csv
import os
import random
import time
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
try:
    import timm
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "timm"])
    import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.filters import threshold_otsu
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", category=UserWarning, module="timm.layers.interpolate")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def resolve_path(candidates, default):
    for c in candidates:
        if c and os.path.exists(c):
            return c
    return default

DATA_DIR = resolve_path([os.getenv("DATA_DIR", ""), "./data", "./dataset", "/kaggle/input/lulc-chennai", "../data"], "./data")
OUT_DIR = os.getenv("OUT_DIR", "./outputs")
os.makedirs(OUT_DIR, exist_ok=True)
NUM_CLASSES = 8
CKPT_NAME = "best_model_final_5.pth"
CVA_WATER_NDWI_THRESHOLD = 0.35
DISPLAY_CONFIDENCE_TAU = 0.60

CONFIG = {
    "img_size": 128,
    "in_chans": 9,
    "num_classes": NUM_CLASSES,
    "batch_size": 32,
    "num_workers": 4,
    "lr": 5.0e-5,
    "weight_decay": 6.358359e-4,
    "focal_gamma": 0.6136,
    "backbone_lr_ratio": 0.2719,
    "epochs": 160,
    "freeze_epochs": 15,
    "early_stop_patience": 20,
    "grad_clip_norm": 1.0,
    "label_smoothing": 0.1,
    "class_weight_cap": 5.0,
    "change_pos_weight_cap": 8.0,
    "change_bce_weight": 0.70,
    "change_dice_weight": 0.30,
    "change_task_weight": 1.0,
    "backbone": "swin_tiny_patch4_window7_224",
    "attn_heads": 8,
    "cross_attn_heads": 8,
    "num_layers": 3,
}

print(f"Device: {device}")

Device: cuda


## Data Loading

In [2]:
patches_2017 = np.load(os.path.join(DATA_DIR, "patches_2017.npy"))
patches_2019 = np.load(os.path.join(DATA_DIR, "patches_2019.npy"))
patches_2024 = np.load(os.path.join(DATA_DIR, "patches_2024.npy"))
patches_2025 = np.load(os.path.join(DATA_DIR, "patches_2025.npy"))
labels = np.load(os.path.join(DATA_DIR, "label_patches.npy"))

YEAR_ARRAYS = {
    2017: patches_2017,
    2019: patches_2019,
    2024: patches_2024,
    2025: patches_2025,
}
LABEL_FOR_YEAR = {year: labels for year in YEAR_ARRAYS}

n_patches = labels.shape[0]
rng = np.random.RandomState(SEED)
shuffled_idx = rng.permutation(n_patches)
n_train = int(n_patches * 0.80)
train_patch_idx = shuffled_idx[:n_train]
val_patch_idx = shuffled_idx[n_train:]
print(f"Split: {len(train_patch_idx)} train / {len(val_patch_idx)} val")


Split: 2774 train / 694 val


## Spectral Processing

In [3]:
def to_nine_band(patch_5band, eps=1e-6):
    b2, b3, b4, b8, b11 = patch_5band[:5]
    ndvi = (b8 - b4) / (b8 + b4 + eps)
    ndwi = (b3 - b8) / (b3 + b8 + eps)
    ndbi = (b11 - b8) / (b11 + b8 + eps)
    mndwi = (b3 - b11) / (b3 + b11 + eps)
    return np.stack([b2, b3, b4, b8, b11, ndvi, ndwi, ndbi, mndwi], axis=0).astype(np.float32)


def normalize_bands(bands):
    out = bands.copy()
    out[:5] = np.clip(out[:5], 0, 1)
    out[5:] = (np.clip(out[5:], -1, 1) + 1) / 2
    return out

## Model Architecture

In [4]:
class UNetDecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch+skip_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),         nn.BatchNorm2d(out_ch), nn.ReLU(True))
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=[1,2,4]):
        super().__init__()
        self.stages = nn.ModuleList([nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))])
        for r in rates:
            self.stages.append(nn.Sequential(
                nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True)))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch*(len(rates)+2),out_ch,1,bias=False),nn.BatchNorm2d(out_ch),nn.ReLU(True))
    def forward(self, x):
        feats = [s(x) for s in self.stages]
        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False)
        feats.append(p)
        return self.project(torch.cat(feats, dim=1))

def widen_patch_embed(backbone, in_chans):
    proj = backbone.patch_embed.proj; old_w = proj.weight.data
    new_proj = nn.Conv2d(in_chans, proj.out_channels, proj.kernel_size, proj.stride, proj.padding,
                         bias=proj.bias is not None)
    with torch.no_grad():
        new_proj.weight[:,:3] = old_w
        if in_chans > 3:
            new_proj.weight[:,3:] = old_w.mean(1,keepdim=True).repeat(1,in_chans-3,1,1)
        if proj.bias is not None: new_proj.bias[:] = proj.bias.data
    backbone.patch_embed.proj = new_proj

def to_nchw(feat, chs):
    if feat.dim()==4 and feat.shape[1]!=feat.shape[-1] and feat.shape[-1] in chs:
        feat = feat.permute(0,3,1,2).contiguous()
    return feat

class BSALayer(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.n1 = nn.LayerNorm(dim); self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.n2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Linear(dim*4,dim))
    def forward(self, x):
        h=self.n1(x); x=x+self.attn(h,h,h,need_weights=False)[0]
        h=self.n2(x); return x+self.mlp(h)

class BottleneckSelfAttention(nn.Module):
    def __init__(self, dim, heads=8, num_layers=1):
        super().__init__()
        self.layers = nn.ModuleList([BSALayer(dim,heads) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(dim)
    def forward(self, x):
        B,C,H,W = x.shape
        t = x.permute(0,2,3,1).reshape(B,H*W,C)
        for l in self.layers: t=l(t)
        return self.norm(t).reshape(B,H,W,C).permute(0,3,1,2)

class CrossTransformerBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.na=nn.LayerNorm(dim); self.nb=nn.LayerNorm(dim)
        self.attn12=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.attn21=nn.MultiheadAttention(dim,heads,batch_first=True)
        self.nc=nn.LayerNorm(dim); self.nd=nn.LayerNorm(dim)
        self.proj=nn.Linear(dim*2,dim)
    def forward(self, f1, f2):
        B,C,H,W = f1.shape
        t1=f1.permute(0,2,3,1).reshape(B,H*W,C); t2=f2.permute(0,2,3,1).reshape(B,H*W,C)
        n1,n2=self.na(t1),self.nb(t2)
        c1=t1+self.attn12(n1,n2,n2,need_weights=False)[0]
        c2=t2+self.attn21(n2,n1,n1,need_weights=False)[0]
        out=self.proj(torch.cat([self.nc(c1),self.nd(c2)],dim=-1))
        return out.reshape(B,H,W,C).permute(0,3,1,2)

class SiameseSwinRefined(nn.Module):
    def __init__(self, num_classes=8, in_chans=9, img_size=128,
                 backbone_name="swin_tiny_patch4_window7_224",
                 pretrained=True, attn_heads=8, cross_heads=8, num_layers=3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained,
                            features_only=True, img_size=img_size, in_chans=3)
        widen_patch_embed(self.backbone, in_chans)
        self.chs = self.backbone.feature_info.channels()

        self.self_attn   = BottleneckSelfAttention(self.chs[3], attn_heads, num_layers)
        self.aspp        = ASPP(self.chs[3], self.chs[3], rates=[1,2,4])
        self.cross_attn  = CrossTransformerBlock(self.chs[3], cross_heads)

        self.dec3 = UNetDecoderBlock(self.chs[3], self.chs[2], self.chs[2])
        self.dec2 = UNetDecoderBlock(self.chs[2], self.chs[1], self.chs[1])
        self.dec1 = UNetDecoderBlock(self.chs[1], self.chs[0], self.chs[0])

        self.lulc_head = nn.Sequential(
            nn.Conv2d(self.chs[0], self.chs[0]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[0]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[0]//2, num_classes, 1))

        self.change_head = nn.Sequential(
            nn.Conv2d(self.chs[3], self.chs[3]//2, 3, padding=1),
            nn.BatchNorm2d(self.chs[3]//2), nn.ReLU(True),
            nn.Conv2d(self.chs[3]//2, 1, 1))

        self.img_size = img_size

    def encode(self, x):
        return [to_nchw(f, self.chs) for f in self.backbone(x)]

    def forward(self, x1, x2):
        f1, f2 = self.encode(x1), self.encode(x2)
        f1a, f2a = self.self_attn(f1[3]), self.self_attn(f2[3])
        f1p, f2p = self.aspp(f1a), self.aspp(f2a)

        s1,s2,s3,_ = f2
        d3 = self.dec3(f2p, s3); d2 = self.dec2(d3,s2); d1 = self.dec1(d2,s1)
        lulc = F.interpolate(self.lulc_head(d1),(self.img_size,self.img_size),mode="bilinear",align_corners=False)

        fused = self.cross_attn(f1p, f2p)
        change = F.interpolate(self.change_head(fused),(self.img_size,self.img_size),mode="bilinear",align_corners=False)
        return lulc, change.squeeze(1)

## Load Checkpoint

In [5]:
model = SiameseSwinRefined(
    num_classes=NUM_CLASSES, in_chans=CONFIG["in_chans"], img_size=CONFIG["img_size"],
    backbone_name=CONFIG["backbone"], pretrained=False,
    attn_heads=CONFIG["attn_heads"], cross_heads=CONFIG["cross_attn_heads"],
    num_layers=CONFIG["num_layers"]
).to(device)

ckpt_path = os.path.join("/kaggle/input/datasets/user/lulc-model-1", CKPT_NAME)
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")


Loaded checkpoint from epoch 158


## TTA

In [6]:
def tta_predict(model, x1, x2):
    lulc_sum = None
    change_sum = None
    for rotation in range(4):
        rotated_x1 = torch.rot90(x1, rotation, (2, 3))
        rotated_x2 = torch.rot90(x2, rotation, (2, 3))
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            lulc_logits, change_logits = model(rotated_x1, rotated_x2)
        lulc_logits = torch.rot90(lulc_logits, -rotation, (2, 3))
        change_logits = torch.rot90(change_logits.unsqueeze(1), -rotation, (2, 3)).squeeze(1)
        lulc_sum = lulc_logits if lulc_sum is None else lulc_sum + lulc_logits
        change_sum = change_logits if change_sum is None else change_sum + change_logits
    return lulc_sum / 4, change_sum / 4

In [7]:
CLASS_NAMES = ["Trees", "Shrubland", "Grassland", "Cropland", "Built-up", "Bare/Sparse", "Water", "Wetland"]

LULC_COLORS = np.array([
    [0, 100, 0], [139, 90, 43], [124, 252, 0], [255, 215, 0],
    [255, 0, 0], [210, 180, 140], [0, 0, 255], [0, 128, 128],
], dtype=np.float32) / 255.0

def label_to_rgb(label_map, colors=LULC_COLORS):
    rgb = np.zeros((*label_map.shape, 3), dtype=np.float32)
    for class_index in range(len(colors)):
        rgb[label_map == class_index] = colors[class_index]
    return rgb

## Grad-CAM Implementation

In [8]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            if isinstance(output, tuple):
                output = output[0]
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, x1, x2, target_class=None):
        self.model.zero_grad()
        lulc_logits, change_logits = self.model(x1, x2)

        # lulc_logits shape: (B, C, H, W)
        # Reduce to one scalar target class per batch item using spatial mode
        if target_class is None:
            # argmax -> (B, H, W), flatten spatial -> (B, H*W), take modal class -> (B,)
            spatial_preds = lulc_logits.argmax(dim=1)          # (B, H, W)
            tc = spatial_preds.view(spatial_preds.shape[0], -1).mode(dim=1).values  # (B,)
        elif isinstance(target_class, int):
            tc = torch.full((lulc_logits.shape[0],), target_class,
                            dtype=torch.long, device=lulc_logits.device)
        else:
            # Assume already a (B,) tensor
            tc = target_class.view(lulc_logits.shape[0]).to(lulc_logits.device)

        # Backprop through the spatially-averaged class score
        spatial_avg = lulc_logits.mean(dim=(2, 3))                  # (B, C)
        scalar_score = spatial_avg.gather(1, tc.unsqueeze(1)).squeeze(1).sum()
        scalar_score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(CONFIG["img_size"], CONFIG["img_size"]),
                            mode="bilinear", align_corners=False)
        cam = cam.squeeze(1).cpu().numpy()

        # Gaussian smoothing removes Swin 7x7 window-border checkerboard noise
        from scipy.ndimage import gaussian_filter
        for b in range(cam.shape[0]):
            cam[b] = gaussian_filter(cam[b], sigma=1.5)
            cam_min, cam_max = cam[b].min(), cam[b].max()
            if cam_max - cam_min > 1e-6:
                cam[b] = (cam[b] - cam_min) / (cam_max - cam_min)
            else:
                cam[b] = 0.0

        return cam, lulc_logits.detach()


## Generate Grad-CAM Visualizations

In [9]:
target_layer = model.dec1.conv
grad_cam = GradCAM(model, target_layer)

sample_indices = val_patch_idx[:6]
sample_pairs = [(2017, 2025), (2019, 2024), (2017, 2024), (2019, 2025), (2024, 2025), (2017, 2019)]

print(f"Generating Grad-CAM for {len(sample_indices)} patches...")

for idx, (patch_idx, (y1, y2)) in enumerate(zip(sample_indices, sample_pairs)):
    img1 = normalize_bands(to_nine_band(YEAR_ARRAYS[y1][patch_idx]))
    img2 = normalize_bands(to_nine_band(YEAR_ARRAYS[y2][patch_idx]))
    rgb1 = np.stack([img1[2], img1[1], img1[0]], axis=-1)
    rgb2 = np.stack([img2[2], img2[1], img2[0]], axis=-1)
    rgb1 = np.clip((rgb1 - np.percentile(rgb1, 2)) / (np.percentile(rgb1, 98) - np.percentile(rgb1, 2) + 1e-6), 0, 1)
    rgb2 = np.clip((rgb2 - np.percentile(rgb2, 2)) / (np.percentile(rgb2, 98) - np.percentile(rgb2, 2) + 1e-6), 0, 1)

    t1 = torch.from_numpy(img1).unsqueeze(0).to(device).requires_grad_(True)
    t2 = torch.from_numpy(img2).unsqueeze(0).to(device).requires_grad_(True)

    cam, logits = grad_cam.generate(t1, t2)
    cam = cam[0]
    pred = logits.argmax(1).cpu().numpy()[0]
    gt = LABEL_FOR_YEAR[y2][patch_idx]

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))

    axes[0, 0].imshow(rgb1)
    axes[0, 0].set_title(f"T1 RGB ({y1})", fontweight="bold")

    axes[0, 1].imshow(rgb2)
    axes[0, 1].set_title(f"T2 RGB ({y2})", fontweight="bold")

    axes[0, 2].imshow(label_to_rgb(gt))
    axes[0, 2].set_title(f"Ground Truth ({y2})", fontweight="bold")

    axes[0, 3].imshow(label_to_rgb(pred))
    axes[0, 3].set_title(f"Prediction ({y2})", fontweight="bold")

    axes[1, 0].imshow(cam, cmap="jet", vmin=0, vmax=1)
    axes[1, 0].set_title("Grad-CAM Heatmap", fontweight="bold")

    overlay = rgb2.copy()
    heatmap = plt.cm.jet(cam)[:, :, :3]
    blended = 0.5 * overlay + 0.5 * heatmap
    axes[1, 1].imshow(np.clip(blended, 0, 1))
    axes[1, 1].set_title("Grad-CAM Overlay on T2", fontweight="bold")

    correct = (pred == gt)
    error_map = np.zeros((*gt.shape, 3), dtype=np.float32)
    error_map[correct] = [0.0, 0.8, 0.0]
    error_map[~correct] = [0.9, 0.0, 0.0]
    axes[1, 2].imshow(error_map)
    axes[1, 2].set_title(f"Error Map (Acc: {correct.mean()*100:.1f}%)", fontweight="bold")

    cam_thresh = (cam > 0.5).astype(np.float32)
    high_attn_acc = correct[cam > 0.5].mean() * 100 if (cam > 0.5).sum() > 0 else 0
    low_attn_acc = correct[cam <= 0.5].mean() * 100 if (cam <= 0.5).sum() > 0 else 0
    labels_text = [f"High attention\n{high_attn_acc:.1f}%", f"Low attention\n{low_attn_acc:.1f}%"]
    bar_vals = [high_attn_acc, low_attn_acc]
    bars = axes[1, 3].bar(labels_text, bar_vals, color=["#FF5722", "#607D8B"], width=0.5)
    axes[1, 3].set_ylim(0, 105)
    axes[1, 3].set_title("Accuracy in Attention Regions", fontweight="bold")
    axes[1, 3].grid(axis="y", alpha=0.3)

    for ax in axes.flat:
        if ax != axes[1, 3]:
            ax.axis("off")

    fig.suptitle(f"Grad-CAM Analysis | Patch {patch_idx} | {y1} → {y2}", fontsize=16, fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"gradcam_patch_{patch_idx}_{y1}_{y2}.png"), dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  [{idx+1}/{len(sample_indices)}] Saved gradcam_patch_{patch_idx}_{y1}_{y2}.png")

print("Grad-CAM generation complete.")

Generating Grad-CAM for 6 patches...
  [1/6] Saved gradcam_patch_933_2017_2025.png
  [2/6] Saved gradcam_patch_928_2019_2024.png
  [3/6] Saved gradcam_patch_823_2017_2024.png
  [4/6] Saved gradcam_patch_1531_2019_2025.png
  [5/6] Saved gradcam_patch_1214_2024_2025.png
  [6/6] Saved gradcam_patch_504_2017_2019.png
Grad-CAM generation complete.


## Per-Class Grad-CAM

In [10]:
patch_idx = val_patch_idx[0]
y1, y2 = 2017, 2025
img1 = normalize_bands(to_nine_band(YEAR_ARRAYS[y1][patch_idx]))
img2 = normalize_bands(to_nine_band(YEAR_ARRAYS[y2][patch_idx]))
rgb2 = np.stack([img2[2], img2[1], img2[0]], axis=-1)
rgb2 = np.clip((rgb2 - np.percentile(rgb2, 2)) / (np.percentile(rgb2, 98) - np.percentile(rgb2, 2) + 1e-6), 0, 1)

t1 = torch.from_numpy(img1).unsqueeze(0).to(device).requires_grad_(True)
t2 = torch.from_numpy(img2).unsqueeze(0).to(device).requires_grad_(True)

fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for c in range(NUM_CLASSES):
    row, col = c // 4, c % 4
    cam, _ = grad_cam.generate(t1, t2, target_class=c)
    cam = cam[0]
    heatmap = plt.cm.jet(cam)[:, :, :3]
    blended = np.clip(0.5 * rgb2 + 0.5 * heatmap, 0, 1)
    axes[row, col].imshow(blended)
    axes[row, col].set_title(f"{CLASS_NAMES[c]}", fontweight="bold", fontsize=12)
    axes[row, col].axis("off")

fig.suptitle(f"Per-Class Grad-CAM | Patch {patch_idx} | {y1} → {y2}", fontsize=16, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f"gradcam_per_class_patch_{patch_idx}.png"), dpi=180, bbox_inches="tight")
plt.show()
print("Saved per-class Grad-CAM")

Saved per-class Grad-CAM
